In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import lightgbm as lgb

# ==========================================
# 1. REPRODUCIBILITY & BUSINESS ASSUMPTIONS (PKR)
# ==========================================
SEED = 42
np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 300
os.makedirs("charts", exist_ok=True)

COMMISSION_RATE = 0.12              # 12% platform fee/margin on completed orders
REVERSE_LOGISTICS_PKR = 450.0       # Two-way shipping + handling loss per failed/returned order
PREVENTIVE_COST_FACTOR = 0.25       # Residual loss/friction when flagged above risk threshold
PREPAID_RETENTION_FACTOR = 0.70     # Retained margin share when risky COD gets nudged to prepaid/verify


# ==========================================
# 2. DOWNLOAD VIA KAGGLEHUB & LOAD DATA
# ==========================================
def load_ecommerce_dataset() -> pd.DataFrame:
    try:
        import kagglehub

        path = kagglehub.dataset_download("zusmani/pakistans-largest-ecommerce-dataset")
        print("Path to dataset files:", path)
        print("Files inside:", os.listdir(path))

        csv_files = [f for f in os.listdir(path) if f.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("No CSV found in downloaded KaggleHub folder.")

        csv_path = os.path.join(path, csv_files[0])
        raw = pd.read_csv(csv_path, low_memory=False)
        print("Raw shape:", raw.shape)
        print("Columns:", raw.columns.tolist())

        # Standardize column lookup safely across raw/cleaned variants
        status_col = "status" if "status" in raw.columns else raw.columns[0]
        total_col = "grand_total" if "grand_total" in raw.columns else "price"
        qty_col = "qty_ordered" if "qty_ordered" in raw.columns else None
        disc_col = "discount_amount" if "discount_amount" in raw.columns else None
        pay_col = "payment_method" if "payment_method" in raw.columns else None
        cat_col = "category_name_1" if "category_name_1" in raw.columns else None
        date_col = "created_at" if "created_at" in raw.columns else None

        valid_statuses = ["complete", "canceled", "order_refunded"]
        df = raw[raw[status_col].isin(valid_statuses)].copy()
        df["target"] = df[status_col].isin(["canceled", "order_refunded"]).astype(int)

        df["order_value"] = pd.to_numeric(df[total_col], errors="coerce").fillna(1500.0).clip(lower=100.0)
        df["qty"] = pd.to_numeric(df[qty_col], errors="coerce").fillna(1.0).clip(lower=1.0) if qty_col else 1.0
        df["discount_amt"] = pd.to_numeric(df[disc_col], errors="coerce").fillna(0.0).clip(lower=0.0) if disc_col else 0.0

        if pay_col:
            payment_clean = df[pay_col].fillna("cod").astype(str).str.lower()
            df["is_cod"] = payment_clean.str.contains("cod|cash").astype(int)
        else:
            df["is_cod"] = 1

        df["category_clean"] = df[cat_col].fillna("Other").astype(str) if cat_col else "Other"

        if date_col:
            created = pd.to_datetime(df[date_col], errors="coerce")
            df["order_hour"] = created.dt.hour.fillna(14).astype(int)
            df["order_dow"] = created.dt.dayofweek.fillna(2).astype(int)
        else:
            df["order_hour"] = 14
            df["order_dow"] = 2

        return df.dropna(subset=["order_value"]).reset_index(drop=True)

    except Exception as exc:
        print(f"KaggleHub load fallback triggered ({exc}). Generating sample data for pipeline test.")
        n = 40000
        is_cod = np.random.binomial(1, 0.78, size=n)
        order_value = np.exp(np.random.normal(7.7, 0.85, size=n)).clip(300, 120000)
        qty = np.random.choice([1, 1, 1, 2, 3, 4], size=n)
        discount_amt = np.where(np.random.rand(n) < 0.35, order_value * np.random.uniform(0.05, 0.35, size=n), 0.0)
        order_hour = np.random.randint(0, 24, size=n)
        order_dow = np.random.randint(0, 7, size=n)
        categories = np.random.choice(
            ["Mobiles & Tablets", "Men's Fashion", "Women's Fashion", "Home & Living", "Appliances"],
            size=n,
            p=[0.22, 0.20, 0.24, 0.18, 0.16],
        )
        logit = (
            -1.65
            + 0.95 * is_cod
            + 0.000018 * order_value
            + 0.85 * (discount_amt / np.maximum(order_value, 1.0))
            + 0.25 * ((order_hour < 7) | (order_hour > 22)).astype(float)
        )
        prob_cancel = 1.0 / (1.0 + np.exp(-logit))
        target = np.random.binomial(1, prob_cancel)

        return pd.DataFrame({
            "target": target,
            "order_value": order_value,
            "qty": qty,
            "discount_amt": discount_amt,
            "is_cod": is_cod,
            "category_clean": categories,
            "order_hour": order_hour,
            "order_dow": order_dow,
        })


df = load_ecommerce_dataset()
print("Filtered modeling rows:", len(df), "| Cancellation/Refund rate:", f"{df['target'].mean():.2%}")


# ==========================================
# 3. FEATURE ENGINEERING & SPLIT
# ==========================================
df["unit_price_est"] = df["order_value"] / df["qty"]
df["discount_ratio"] = np.where(
    (df["order_value"] + df["discount_amt"]) > 0,
    df["discount_amt"] / (df["order_value"] + df["discount_amt"]),
    0.0,
)
df["is_late_night"] = ((df["order_hour"] < 7) | (df["order_hour"] >= 23)).astype(int)
df["cat_code"] = df["category_clean"].astype("category").cat.codes

feature_cols = [
    "is_cod",
    "order_value",
    "qty",
    "unit_price_est",
    "discount_amt",
    "discount_ratio",
    "order_hour",
    "order_dow",
    "is_late_night",
    "cat_code",
]

X = df[feature_cols]
y = df["target"].values

X_train, X_val, y_train, y_val, _, val_order_vals = train_test_split(
    X, y, df["order_value"].values, test_size=0.2, random_state=SEED, stratify=y
)


# ==========================================
# 4. TRAIN LIGHTGBM & EVALUATE
# ==========================================
model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=SEED,
    verbose=-1,
)
model.fit(X_train, y_train)

val_probs = model.predict_proba(X_val)[:, 1]
roc_auc = roc_auc_score(y_val, val_probs)
pr_auc = average_precision_score(y_val, val_probs)
print(f"Validation ROC-AUC: {roc_auc:.4f}")
print(f"Validation PR-AUC:  {pr_auc:.4f}")


# ==========================================
# 5. PROFIT SIMULATION (BEFORE VS. AFTER AI)
# ==========================================
COMMISSION_RATE = 0.12
REVERSE_LOGISTICS_PKR = 450.0
PREVENTIVE_COST_FACTOR = 0.25
PREVENTIVE_RETENTION_FACTOR = 0.70


def evaluate_threshold(y_true, probs, amounts, threshold, return_cost):
    flagged = probs >= threshold
    gross_profit = 0.0
    total_loss = 0.0

    for actual_cancel, is_flagged, amt in zip(y_true, flagged, amounts):
        if not is_flagged:
            if actual_cancel == 1:
                total_loss += return_cost
            else:
                gross_profit += amt * COMMISSION_RATE
        else:
            if actual_cancel == 1:
                total_loss += return_cost * PREVENTIVE_COST_FACTOR
            else:
                gross_profit += amt * COMMISSION_RATE * PREVENTIVE_RETENTION_FACTOR

    net_profit = gross_profit - total_loss
    return gross_profit, total_loss, net_profit
def evaluate_threshold(y_true, probs, amounts, threshold, return_cost):
    flagged = probs >= threshold
    gross_profit = 0.0
    total_loss = 0.0

    for actual_cancel, is_flagged, amt in zip(y_true, flagged, amounts):
        if not is_flagged:
            if actual_cancel == 1:
                total_loss += return_cost
            else:
                gross_profit += amt * COMMISSION_RATE
        else:
            if actual_cancel == 1:
                total_loss += return_cost * PREVENTIVE_COST_FACTOR
            else:
                gross_profit += amt * COMMISSION_RATE * PREVENTIVE_RETENTION_FACTOR

    net_profit = gross_profit - total_loss
    return gross_profit, total_loss, net_profit


thresholds = np.linspace(0.15, 0.85, 71)
best_th = max(
    thresholds,
    key=lambda t: evaluate_threshold(y_val, val_probs, val_order_vals, t, REVERSE_LOGISTICS_PKR)[2],
)

before_gp, before_loss, before_net = evaluate_threshold(
    y_val, val_probs, val_order_vals, 1.01, REVERSE_LOGISTICS_PKR
)
after_gp, after_loss, after_net = evaluate_threshold(
    y_val, val_probs, val_order_vals, best_th, REVERSE_LOGISTICS_PKR
)

print(f"Optimal Threshold:          {best_th:.2f}")
print(f"Before AI Net Profit (PKR): {before_net:,.0f}")
print(f"After  AI Net Profit (PKR): {after_net:,.0f}")
print(f"Net Gain (PKR):             {after_net - before_net:+,.0f}")


# ==========================================
# 6. GENERATE & SAVE SLIDE CHARTS
# ==========================================
# Chart A: Predicted cancellation probability distribution
plt.figure(figsize=(7, 4.2))
plt.hist(val_probs[y_val == 0], bins=40, alpha=0.65, label="Completed (0)", color="#2ca02c")
plt.hist(val_probs[y_val == 1], bins=40, alpha=0.65, label="Canceled/Refunded (1)", color="#d62728")
plt.axvline(best_th, color="#1f77b4", linestyle="--", linewidth=2, label=f"Threshold = {best_th:.2f}")
plt.title("Predicted Risk Distribution & Cutoff Threshold")
plt.xlabel("Predicted Cancellation / Return Probability")
plt.ylabel("Order Count")
plt.legend()
plt.tight_layout()
plt.savefig("charts/risk_distribution.png")
plt.close()

# Chart B: Before vs. After Net Profit (in Million PKR)
labels = ["Before AI", "After AI"]
net_vals_m = [before_net / 1e6, after_net / 1e6]
loss_vals_m = [before_loss / 1e6, after_loss / 1e6]

x_idx = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(6.5, 4.2))
plt.bar(x_idx - width / 2, net_vals_m, width, label="Net Profit (M PKR)", color="#1f77b4")
plt.bar(x_idx + width / 2, loss_vals_m, width, label="Logistics Loss (M PKR)", color="#ff7f0e")
plt.xticks(x_idx, labels)
plt.ylabel("Amount (Million PKR)")
plt.title("Simulated Net Profit & Logistics Loss: Before vs. After AI")
plt.legend()
plt.tight_layout()
plt.savefig("charts/profit_simulation.png")
plt.close()

# Chart C: Top 4 Feature Importances
importances = (
    pd.Series(model.feature_importances_, index=feature_cols)
    .sort_values(ascending=True)
    .tail(4)
)
plt.figure(figsize=(6.5, 4.0))
importances.plot(kind="barh", color="#2b5c8f")
plt.title("Top 4 Features Driving Risk Predictions")
plt.xlabel("Split Importance (LightGBM)")
plt.tight_layout()
plt.savefig("charts/top_features.png")
plt.close()

print("Saved charts to ./charts/: risk_distribution.png, profit_simulation.png, top_features.png")

Using Colab cache for faster access to the 'pakistans-largest-ecommerce-dataset' dataset.
Path to dataset files: /kaggle/input/pakistans-largest-ecommerce-dataset
Files inside: ['Pakistan Largest Ecommerce Dataset.csv']
Raw shape: (1048575, 26)
Columns: ['item_id', 'status', 'created_at', 'sku', 'price', 'qty_ordered', 'grand_total', 'increment_id', 'category_name_1', 'sales_commission_code', 'discount_amount', 'payment_method', 'Working Date', 'BI Status', ' MV ', 'Year', 'Month', 'Customer Since', 'M-Y', 'FY', 'Customer ID', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25']
Filtered modeling rows: 494463 | Cancellation/Refund rate: 52.74%
Validation ROC-AUC: 0.8012
Validation PR-AUC:  0.8273
Optimal Threshold:          0.22
Before AI Net Profit (PKR): 3,499,494
After  AI Net Profit (PKR): 12,954,085
Net Gain (PKR):             +9,454,591
Saved charts to ./charts/: risk_distribution.png, profit_simulation.png, top_features.png


In [4]:
import matplotlib.pyplot as plt
import numpy as np

# 1. DONUT CHART: Order Status Share (like Slide 10 in Sample2)
status_counts = df["target"].value_counts().sort_index()
labels = ["Completed (47.3%)", "Canceled / Refunded (52.7%)"]
colors = ["#2ca02c", "#d62728"]

plt.figure(figsize=(5.5, 5.5))
wedges, texts = plt.pie(
    status_counts,
    labels=labels,
    colors=colors,
    startangle=90,
    wedgeprops=dict(width=0.42, edgecolor="white", linewidth=2),
    textprops=dict(fontsize=11, fontweight="bold"),
)
plt.title("Order Outcome Breakdown (494,463 Orders)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("charts/status_donut.png", dpi=300)
plt.close()

# 2. TOP 4 FEATURE TREND DISTRIBUTIONS (like Slide 8 in Sample2)
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))

plot_specs = [
    ("order_value", "Order Value (PKR)", (100, 25000)),
    ("discount_ratio", "Discount Ratio", (0.0, 0.5)),
    ("order_hour", "Order Hour (0-23)", (0, 23)),
    ("is_cod", "Payment (0=Prepaid, 1=COD)", (-0.2, 1.2)),
]

for ax, (col, title, xlim) in zip(axes, plot_specs):
    comp = df.loc[df["target"] == 0, col].clip(xlim[0], xlim[1])
    canc = df.loc[df["target"] == 1, col].clip(xlim[0], xlim[1])
    ax.hist(comp, bins=25, density=True, alpha=0.55, color="#2ca02c", label="Completed")
    ax.hist(canc, bins=25, density=True, alpha=0.55, color="#d62728", label="Canceled")
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlim(xlim)
    ax.grid(axis="y", linestyle=":", alpha=0.5)

axes[0].set_ylabel("Density", fontsize=11)
axes[-1].legend(loc="upper left", fontsize=9)
plt.suptitle("Top Features: Completed (Green) vs. Canceled/Refunded (Red)", fontsize=13, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig("charts/feature_trends_4panel.png", dpi=300, bbox_inches="tight")
plt.close()

print("Saved: charts/status_donut.png and charts/feature_trends_4panel.png")

Saved: charts/status_donut.png and charts/feature_trends_4panel.png
